In [1]:
!pip install sastrawi tensorflow

import numpy as np
import pickle
import re
import string
import nltk
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.6 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
import pickle
from tensorflow.keras.models import load_model

model_lstm = load_model('model_lstm.h5')
model_gru  = load_model('model_gru.h5')
model_cnn  = load_model('model_cnn.h5')

with open('tfidf_lstm.pkl', 'rb') as f:
    tfidf_lstm = pickle.load(f)

with open('tfidf_gru.pkl', 'rb') as f:
    tfidf_gru = pickle.load(f)

with open('tokenizer_cnn.pkl', 'rb') as f:
    tokenizer_cnn = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

MAX_LENGTH = 50
print("Semua model dan tools berhasil diload")

Semua model dan tools berhasil diload


In [3]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

slangwords = {
    'gak':'tidak','ga':'tidak','ngak':'tidak','nggak':'tidak','gk':'tidak','tdk':'tidak',
    'ga bisa':'tidak bisa','gamau':'tidak mau','gabisa':'tidak bisa',

    'udah':'sudah','udh':'sudah','dah':'sudah','sdh':'sudah','uda':'sudah',
    'blm':'belum','blom':'belum','belom':'belum',

    'bgt':'sangat','bngt':'sangat','banget':'sangat','bener bener':'sungguh',
    'mantap':'bagus','mantul':'bagus','keren':'bagus','ok':'bagus','oke':'bagus',
    'okelah':'bagus','ok lah':'bagus','top':'bagus','jos':'bagus',
    'josss':'bagus','joss':'bagus','asyik':'bagus','asik':'bagus',
    'sip':'bagus','sippp':'bagus','sippp':'bagus','kece':'bagus',
    'ciamik':'bagus','canggih':'bagus','kinclong':'bagus',
    'lancar':'lancar','cepat':'cepat','mudah':'mudah','gampang':'mudah',

    'lelet':'lambat','lemot':'lambat','lambat':'lambat',
    'ribet':'rumit','ruwet':'rumit','susah':'sulit','susah':'sulit',
    'eror':'error','err':'error','hang':'error','ngehang':'error',
    'bug':'error','bugg':'error','buggy':'error',
    'parah':'buruk','paling parah':'sangat buruk','jelek':'buruk',
    'jelekk':'buruk','ancur':'buruk','amburadul':'buruk',
    'zonk':'buruk','zonkk':'buruk','kacau':'buruk',
    'ngecewain':'mengecewakan','nyebelin':'menjengkelkan',
    'nyusahin':'menyusahkan','ngeselin':'menjengkelkan',
    'ganggu':'mengganggu','ngganggu':'mengganggu',

    'maks':'maksimal','lbh':'lebih',
    'bener':'benar','bnr':'benar','beneran':'benar',
    'emg':'memang','emang':'memang','mmg':'memang',
    'kyk':'seperti','kyak':'seperti',
    'gini':'begini','gitu':'begitu',
    'tp':'tapi','tpi':'tapi',
    'kl':'kalau','klo':'kalau','klu':'kalau','kalo':'kalau',
    'krn':'karena','karna':'karena','krna':'karena',
    'pdhl':'padahal','pdhal':'padahal',
    'jg':'juga','trs':'terus','trus':'terus',
    'aja':'saja','aj':'saja','doang':'saja',
    'nih':'ini','ni':'ini','tu':'itu','tuh':'itu',
    'gw':'saya','gue':'saya','w':'saya',
    'lo':'kamu','lu':'kamu','elu':'kamu',
    'bs':'bisa','dr':'dari','utk':'untuk','tuk':'untuk',
    'dgn':'dengan','dg':'dengan','yg':'yang',
    'sm':'sama','ama':'sama','lg':'lagi','lgi':'lagi',
    'dpt':'dapat','dpat':'dapat','hrs':'harus',
    'mkn':'mungkin','mgkn':'mungkin','mo':'mau',
    'dtg':'datang','pake':'pakai','make':'pakai',
    'balikin':'kembalikan','benerin':'perbaiki','beresin':'perbaiki',
    'makin':'semakin','tau':'tahu','taw':'tahu',
    'knp':'kenapa','knpa':'kenapa',
    'cuma':'hanya','cm':'hanya','cmn':'hanya',
    'kmrn':'kemarin','skrg':'sekarang','skrng':'sekarang',
    'sbnrnya':'sebenarnya','sebenernya':'sebenarnya',
    'makasih':'terima kasih','thx':'terima kasih','tks':'terima kasih',
    'notif':'notifikasi','trf':'transfer',

    'm-banking':'mobile banking','mbanking':'mobile banking',
    'inet':'internet','inet banking':'internet banking',
    'atm':'atm','wd':'tarik tunai','topup':'isi saldo',
    'login':'masuk','logout':'keluar','log in':'masuk',
    'reload':'muat ulang','refresh':'segarkan',
    'transaksi':'transaksi','verif':'verifikasi','otp':'kode otp'
}

def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    text = re.sub(r'#[A-Za-z0-9]+', '', text)
    text = re.sub(r'RT[\s]', '', text)
    text = re.sub(r"http\S+", '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.replace('\n', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip(' ')
    return text

def casefoldingText(text):
    return text.lower()

def fix_slangwords(text):
    words = text.split()
    return ' '.join([slangwords[w.lower()] if w.lower() in slangwords else w for w in words])

def tokenizingText(text):
    return word_tokenize(text)

def filteringText(text):
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya', 'yaa', 'nya', 'na', 'sih', 'ku', 'di', 'ya', 'loh', 'kah', 'woi', 'woii', 'woy', 'aja',
                          'udah', 'udh', 'terus', 'pas', 'deh', 'dong', 'nih', 'tuh', 'nihh', 'tuhh', 'doang', 'saja',
                          'lah', 'tah', 'kan', 'pun', 'kok', 'eh', 'hmm'])
    return [w for w in text if w not in listStopwords]

def stemmingText(text):
    words = text.split()
    return ' '.join([stemmer.stem(w) for w in words])

def toSentence(list_words):
    return ' '.join(word for word in list_words)

def preprocessing_inference(teks):
    teks = cleaningText(teks)
    teks = casefoldingText(teks)
    teks = fix_slangwords(teks)
    teks = tokenizingText(teks)
    teks = filteringText(teks)
    teks = toSentence(teks)
    teks = stemmingText(teks)
    return teks

In [4]:
def prediksi_sentimen(teks_input):
    print(f"\nTeks Asli   : {teks_input}")
    teks_clean = preprocessing_inference(teks_input)
    print(f"Teks Bersih : {teks_clean}")
    print("-" * 55)

    # LSTM + TF-IDF
    X_lstm     = tfidf_lstm.transform([teks_clean]).toarray()
    X_lstm_3d  = X_lstm.reshape(X_lstm.shape[0], 1, X_lstm.shape[1])
    pred_lstm  = model_lstm.predict(X_lstm_3d, verbose=0)
    label_lstm = le.inverse_transform([np.argmax(pred_lstm)])[0]
    conf_lstm  = np.max(pred_lstm) * 100

    # GRU + TF-IDF
    X_gru     = tfidf_gru.transform([teks_clean]).toarray()
    X_gru_3d  = X_gru.reshape(X_gru.shape[0], 1, X_gru.shape[1])
    pred_gru  = model_gru.predict(X_gru_3d, verbose=0)
    label_gru = le.inverse_transform([np.argmax(pred_gru)])[0]
    conf_gru  = np.max(pred_gru) * 100

    # CNN + FastText
    X_cnn_seq  = tokenizer_cnn.texts_to_sequences([teks_clean])
    X_cnn_pad  = pad_sequences(X_cnn_seq, maxlen=MAX_LENGTH, padding='post')
    pred_cnn   = model_cnn.predict(X_cnn_pad, verbose=0)
    label_cnn  = le.inverse_transform([np.argmax(pred_cnn)])[0]
    conf_cnn   = np.max(pred_cnn) * 100

    print(f"[LSTM + TF-IDF]    Prediksi: {label_lstm.upper():<10} | Confidence: {conf_lstm:.2f}%")
    print(f"[GRU  + TF-IDF]    Prediksi: {label_gru.upper():<10} | Confidence: {conf_gru:.2f}%")
    print(f"[CNN  + FastText]   Prediksi: {label_cnn.upper():<10} | Confidence: {conf_cnn:.2f}%")

In [5]:
teks_uji = [
    "aplikasi bca sering error dan lambat banget",
    "transfer uang mudah dan cepat, suka banget sama bca",
    "aplikasi mudah dipakai namun fitur tabungan masih terbatas",
    "aplikasi berjalan lancar namun fitur masih terbatas",
    "transfer antar bank berhasil tetapi biaya admin terasa memberatkan"
]

for teks in teks_uji:
    prediksi_sentimen(teks)
    print()


Teks Asli   : aplikasi bca sering error dan lambat banget
Teks Bersih : aplikasi bca error lambat
-------------------------------------------------------
[LSTM + TF-IDF]    Prediksi: NEGATIVE   | Confidence: 100.00%
[GRU  + TF-IDF]    Prediksi: NEGATIVE   | Confidence: 99.66%
[CNN  + FastText]   Prediksi: NEGATIVE   | Confidence: 100.00%


Teks Asli   : transfer uang mudah dan cepat, suka banget sama bca
Teks Bersih : transfer uang mudah cepat suka bca
-------------------------------------------------------
[LSTM + TF-IDF]    Prediksi: POSITIVE   | Confidence: 99.35%
[GRU  + TF-IDF]    Prediksi: POSITIVE   | Confidence: 87.82%
[CNN  + FastText]   Prediksi: POSITIVE   | Confidence: 100.00%


Teks Asli   : aplikasi mudah dipakai namun fitur tabungan masih terbatas
Teks Bersih : aplikasi mudah pakai fitur tabung batas
-------------------------------------------------------
[LSTM + TF-IDF]    Prediksi: NEGATIVE   | Confidence: 68.96%
[GRU  + TF-IDF]    Prediksi: NEGATIVE   | Confidence: 6